# Day 11 — CuTe DSL Layout Algebra

**Goal:** internalize the four operations every later DSL kernel uses:
constructing layouts, `zipped_divide`, Thread-Value (TV) layouts, and
identity tensors for predication.

A **Layout** = (Shape, Stride) — a pure function from coordinates →
linear offset. The DSL exposes the same algebra as the C++ side, but at
JIT trace time you can just `print(layout)` and read what you built.

No GPU kernels in this notebook — pure layout exercises.


In [ ]:
import cutlass
import cutlass.cute as cute

cutlass.cuda.initialize_cuda_context()


## Part 1 — row-major vs. column-major

Build two `(4, 8)` layouts with strides `(N, 1)` (row-major) and
`(1, M)` (column-major). Verify `size == M * N`.


In [ ]:
@cute.jit
def part1_build_layouts():
    M, N = 4, 8
    # TODO: build row_major (stride=(N,1)) and col_major (stride=(1,M)),
    #       print both, assert size(row_major) == M*N.
    raise NotImplementedError


# part1_build_layouts()


## Part 2 — `zipped_divide`, the bread-and-butter tiler

`zipped_divide(A, tile)` splits an `(M, N)` layout into
`((TileM, TileN), (NumTilesM, NumTilesN))`. Used by every later kernel
to grab "the bidx-th tile" with `gA[((None, None), bidx)]`.


In [ ]:
@cute.jit
def part2_zipped_divide():
    A = cute.make_layout((8, 16), stride=(16, 1))
    tile = (2, 4)
    # TODO: apply zipped_divide, print result, assert shape ((2,4), (4,4)).
    raise NotImplementedError


# part2_zipped_divide()


## Part 3 — Thread-Value (TV) layout

`(tid, vid) -> tile coord`, the canonical CuTe partitioning. Build
`thr_layout = (4, 32):(32, 1)` (128 threads, contiguous on N) and
`val_layout = (4, 4):(4, 1)` (each thread owns 4×4 elements), then call
`cute.make_layout_tv`.


In [ ]:
@cute.jit
def part3_tv_layout():
    # TODO: build thr_layout, val_layout, call make_layout_tv, print + assert.
    raise NotImplementedError


# part3_tv_layout()


## Part 4 — identity coordinate tensor

`make_identity_tensor(shape)` is a tensor where `T(c) = c` — used on
day 13 to build OOB predicates without writing index math by hand.


In [ ]:
@cute.jit
def part4_identity_tensor():
    # TODO: make_identity_tensor((3, 4)); print cT[0], cT[1], cT[(2, 3)].
    raise NotImplementedError


# part4_identity_tensor()


---

## Reference solution


In [ ]:
@cute.jit
def part1_solution():
    M, N = 4, 8
    row_major = cute.make_layout((M, N), stride=(N, 1))
    col_major = cute.make_layout((M, N), stride=(1, M))
    print(f"row_major = {row_major}")
    print(f"col_major = {col_major}")
    print(f"size      = {cute.size(row_major)}  cosize = {cute.cosize(row_major)}")
    assert cute.size(row_major) == M * N


@cute.jit
def part2_solution():
    A = cute.make_layout((8, 16), stride=(16, 1))
    divided = cute.zipped_divide(A, (2, 4))
    print(f"A       = {A}")
    print(f"divided = {divided}")
    assert tuple(divided.shape[0]) == (2, 4)
    assert tuple(divided.shape[1]) == (4, 4)


@cute.jit
def part3_solution():
    thr_layout = cute.make_layout((4, 32), stride=(32, 1))
    val_layout = cute.make_layout((4, 4),  stride=(4, 1))
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)
    print(f"tiler_mn  = {tiler_mn}")
    print(f"tv_layout = {tv_layout}")
    assert cute.size(tv_layout) == 128 * 16


@cute.jit
def part4_solution():
    cT = cute.make_identity_tensor((3, 4))
    print(f"type      = {cT.type}")
    print(f"cT[0]     = {cT[0]}")
    print(f"cT[1]     = {cT[1]}")
    print(f"cT[(2,3)] = {cT[(2, 3)]}")


part1_solution()
part2_solution()
part3_solution()
part4_solution()
